In [27]:
#Loading groq_key
from dotenv import load_dotenv
import os
load_dotenv()



True

In [28]:
#Loading Documents
from langchain_community.document_loaders import PyPDFLoader
pdf_loader=PyPDFLoader("resources/Java.pdf")

docs=pdf_loader.load()


In [29]:
#Splitting the documents into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=400,chunk_overlap=80)

splitted_docs=splitter.split_documents(docs)


In [41]:
#Converting the chunks of text into embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)




Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7090.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
#Using Chroma DB to store the embeddings

from langchain_community.vectorstores import Chroma

vector_db=Chroma.from_documents(splitted_docs,embedding)

print(vector_db)

In [42]:
#Using Retriever to retrieve the results

user_query="What is Java?"
retriever=vector_db.as_retriever(search_kwargs={"k":3})


In [43]:
#Augumenting the result to the user prompt
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain


prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Use the following context to answer the question.

Context:
{context}

Question:
{input}

Answer:
""")

llm=ChatGroq(model="Llama-3.1-8B-instant")
prompt_chain=create_stuff_documents_chain(llm,prompt)
retrieval_chain=create_retrieval_chain(retriever, prompt_chain)

retrieval_chain.invoke({"input":"What is java?"})



{'input': 'What is java?',
 'context': [Document(metadata={'source': 'resources/Java.pdf', 'author': 'MRCET', 'creator': 'Microsoft® Word 2016', 'moddate': '2021-08-03T07:36:44+00:00', 'page_label': '10', 'creationdate': '2021-08-03T07:36:44+00:00', 'producer': 'www.ilovepdf.com', 'page': 9, 'total_pages': 115}, page_content='suited for internet programming. Later, Java technology as incorporated by \nNetscape. \n \nCurrently, Java is used in internet programming, mobile devices, games, e -business \nsolutions etc. There are given the major points that describes the history of java. \n \n1) James Gosling, Mike Sheridan, and Patrick Naughton initiated the Java'),
  Document(metadata={'page': 9, 'moddate': '2021-08-03T07:36:44+00:00', 'author': 'MRCET', 'source': 'resources/Java.pdf', 'creator': 'Microsoft® Word 2016', 'page_label': '10', 'creationdate': '2021-08-03T07:36:44+00:00', 'producer': 'www.ilovepdf.com', 'total_pages': 115}, page_content='suited for internet programming. Later,